# NeuroFramework 3D-sMRI: Dedicated Axial 3D Model Training
### ABIDE-I Multi-Site Cohort ($N=395$: NYU, UM_1, USM)

This notebook trains the **Dedicated Axial 3D Deep Convolutional Neural Network (Conv3D-CBAM)** on **Lightning AI Studio (Free Tier)** with an NVIDIA T4 or L4 GPU.

#### What This Pipeline Does Automatically:
1. Fetches official ABIDE-I structural MRI scans directly from the AWS S3 neuroimaging archive.
2. Executes Otsu Adaptive Brain Masking, N4 Bias Field Correction, and Site-Harmonized Normalization via multi-core parallel processing.
3. Extracts 50-slice Axial 3D Volumetric Tensors at Full HD ($224 \times 224$).
4. Trains the 5-stage residual Conv3D network with 3D CBAM attention across a **Stratified 5-Fold Cross Validation Protocol** using Mixed-Precision (`torch.cuda.amp`).
5. Automatically saves the winning checkpoint to `checkpoints/best_axial_3d_model.pt` and packages a ready-to-download archive.

## 1. Hardware Telemetry & Accelerator Check

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Accelerator: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Install Project Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Run End-to-End Cloud Training Pipeline
This runs multi-threaded S3 acquisition, parallel CPU preprocessing, and 5-Fold Cross-Validation on the GPU.

In [ ]:
!python training/run_cloud_pipeline.py --model axial --epochs 40 --batch-size 8

## 4. Inspect Cross-Validation Results

In [ ]:
import json
import os
import pandas as pd

metrics_file = "checkpoints/metrics_axial_3d.json"
if os.path.exists(metrics_file):
    with open(metrics_file, "r") as f:
        data = json.load(f)
    
    print("="*60)
    print("DEDICATED AXIAL 3D MODEL: 5-FOLD STRATIFIED BENCHMARK")
    print("="*60)
    df_res = pd.DataFrame(data["folds"])
    print(df_res[["fold", "accuracy", "auc", "sensitivity", "specificity"]].to_string(index=False))
    print("-"*60)
    print(f"MEAN ACCURACY    : {data['mean_accuracy']*100:.2f}%")
    print(f"MEAN ROC-AUC     : {data['mean_auc']:.4f}")
    print(f"MEAN SENSITIVITY : {data['mean_sensitivity']*100:.2f}%")
    print(f"MEAN SPECIFICITY : {data['mean_specificity']*100:.2f}%")
    print(f"BEST FOLD        : Fold {data['best_fold']}")
    print("="*60)
else:
    print("Metrics file not found yet. Run the training cell above first.")

## 5. Download Trained Weights to Your Local Machine
Right-click `trained_weights_axial_3d.zip` in the file explorer on the left and select **Download**.
Extract and place `best_axial_3d_model.pt` into your local `checkpoints/` directory on your laptop.

In [ ]:
import os
zip_name = "trained_weights_axial_3d.zip"
if os.path.exists(zip_name):
    print(f"Archive ready for download: {zip_name} ({os.path.getsize(zip_name)/1e6:.2f} MB)")
    try:
        from google.colab import files
        files.download(zip_name)
    except ImportError:
        print("Running on Lightning AI / Local: Right-click the file in the sidebar and click Download.")